# Portable local backup and fresh restore

**Inputs:** completed v3 scientific run IDs, full trajectory archive, curriculum and notebooks. **Outputs:** SHA256-verified compressed archive under `exports/bundles/`, isolated restored workspace and fresh-kernel inference receipt for every model. **Runtime:** minutes, proportional to archived trajectories. **Default Run All:** definition/readiness inspection only. The controller calls `v3_export_experiment` after writers have finished. **Resume:** restart the v3 controller; a committed receipt is hash-checked and reused. An incomplete unreceipted restore is never silently reused. The user deferred off-VM copying.

The archive includes exact training recovery state (optimizer, predictor, EMA, sampler/replay and RNG) and immutable trajectory dependencies. Restored inference is tested independently; continuing completed training requires an explicitly declared compatible continuation schedule.

The environment lock and pinned GNU Go evaluation binary are included. Python/Torch and the remaining third-party environment must be installed from the recorded lock on another machine; the isolated restore check uses this VM's existing project kernel.

In [ ]:
from pathlib import Path
import nbformat,os
PROJECT=Path(os.environ.get('AHRV3_WORKSPACE',Path.cwd())).resolve()
while not (PROJECT/'notebooks/00_instance_setup_and_recovery.ipynb').is_file():
    if PROJECT.parent==PROJECT:raise RuntimeError('Run within AHR Go workspace')
    PROJECT=PROJECT.parent
for cell in nbformat.read(PROJECT/'experiments/ahr_jepa_v3/00_shared.ipynb',4).cells:
    if cell.cell_type=='code' and 'ahr-export' in cell.metadata.get('tags',[]):exec(compile(cell.source,'v3-shared:export','exec'),globals())
v3_load_upstream(PROJECT,globals())
for name in ('08_predictive_ahr.ipynb','09_trajectory_learning.ipynb'):
    v3_import_exports(PROJECT/V3_ROOT/name,globals())


In [ ]:
import tarfile, tempfile, shutil, subprocess, sys

def v3_bundle_sources(project):
    project=Path(project)
    return {str(path.relative_to(project)):v3_code_sha(path) for path in sorted((project/V3_ROOT).glob('*.ipynb'))}

def v3_export_experiment(project, run_ids, label='final'):
    project=Path(project).resolve()
    if not run_ids or len(set(run_ids))!=len(run_ids):raise ValueError('Unique nonempty run list required')
    for run in run_ids:
        if Path(run).name!=run:raise ValueError('Simple run IDs only')
    root=project/'exports/bundles';root.mkdir(parents=True,exist_ok=True)
    receipt_path=v3_report_path(project,'local_bundle_'+label+'.json')
    if receipt_path.exists():
        receipt=v3_read_json(receipt_path)
        if receipt['run_ids']!=list(run_ids):raise RuntimeError('Existing bundle identifies another experiment')
        if file_sha256(project/receipt['archive'])!=receipt['archive_sha256']:raise RuntimeError('Existing archive hash mismatch')
        return receipt
    # Called only after all training/evaluations finish, from the single controller.
    paths=set()
    for directory in (project/'notebooks',project/V3_ROOT,project/'reports/ahr_jepa_v3'):
        paths.update(p for p in directory.rglob('*') if p.is_file() and p.suffix in ('.ipynb','.json','.jsonl','.gz','.png','.md') and '.ipynb_checkpoints' not in p.parts and not (directory.name=='ahr_jepa_v3' and directory.parent.name=='reports' and p.name in ('progress.json','progress.png')))
    paths.update(p for p in (project/'AGENTS.md',project/'README.md',project/'config/ahr_jepa_v3.json',project/'config/go_generations_v2.json') if p.is_file())
    paths.add(project/'reports/09_13_training_pipeline.ipynb')
    # Close the provenance dependency graph used by shared00 and restored arenas.
    prior_protocol=v3_read_json(project/'config/go_generations_v2.json')
    for relative in prior_protocol['sources']:
        path=project/relative
        if not path.is_file() or not path.resolve().is_relative_to(project):raise RuntimeError('Missing or external frozen source: '+relative)
        paths.add(path)
    paths.update(p for p in (project/'config').glob('*') if p.is_file() and p.suffix in ('.json','.txt','.toml'))
    theory=project/'AHR_Unified_Theory (1).pdf'
    if theory.is_file():paths.add(theory)
    environment=project/'reports/00_environment.json'
    if environment.exists():paths.add(environment)
    reference_binary=project/'.venv/third_party/gnugo/usr/bin/gnugo'
    if reference_binary.exists():paths.add(reference_binary)
    dependencies=[V3_BASELINE,*run_ids]
    dependencies.extend(p.name for p in (project/'checkpoints').glob('arena-v3-*') if p.is_dir())
    for run in dependencies:
        directory=project/'checkpoints'/run
        manifest=v3_read_json(directory/'manifest.json')
        paths.add(directory/'manifest.json')
        for companion in ('identity.json','summary.json'):
            if (directory/companion).is_file():paths.add(directory/companion)
        if run.startswith('arena-v3-') and not (directory/'identity.json').is_file():raise RuntimeError('Arena identity dependency missing: '+run)
        for entry in manifest['entries']:
            path=directory/entry['file']
            if file_sha256(path)!=entry['sha256']:raise RuntimeError('Checkpoint hash mismatch: '+str(path))
            paths.add(path)
    inventory={str(path.relative_to(project)):{'sha256':file_sha256(path),'bytes':path.stat().st_size} for path in sorted(paths)}
    manifest={'schema':'ahr_v3_bundle','run_ids':list(run_ids),'files':inventory,'source_identity':v3_source_identity(project),
              'scope':'All declared training run recovery payloads, EMA/predictor/optimizer/replay archive, curriculum, frozen baseline, experiment notebooks and reports.',
              'off_instance_backup':'DEFERRED_BY_USER'}
    digest=v3_digest(manifest);archive=root/f'ahr-jepa-v3-{label}-{digest[:12]}.tar.gz'
    with tempfile.TemporaryDirectory(dir=root,prefix='.bundle-') as scratch:
        scratch=Path(scratch);manifest_path=scratch/'bundle_manifest.json';atomic_json(manifest_path,manifest)
        temporary=scratch/'bundle.tar.gz'
        with tarfile.open(temporary,'w:gz') as handle:
            handle.add(manifest_path,arcname='bundle_manifest.json',recursive=False)
            for path in sorted(paths):handle.add(path,arcname=str(path.relative_to(project)),recursive=False)
        with open(temporary,'rb') as handle:os.fsync(handle.fileno())
        os.replace(temporary,archive)
    digest_archive=file_sha256(archive)
    restored=project/'exports/restored'/f'ahr-jepa-v3-{label}-{digest[:12]}'
    if restored.exists():raise RuntimeError('Unreceipted restore directory already exists; inspect it')
    restored.mkdir(parents=True)
    with tarfile.open(archive,'r:gz') as handle:
        members=handle.getmembers()
        if len(members)!=len(inventory)+1 or len({m.name for m in members})!=len(members):raise RuntimeError('Archive member inventory mismatch')
        for member in members:
            target=restored/member.name
            if not member.isfile() or member.name.startswith('/') or '..' in Path(member.name).parts or not target.resolve().is_relative_to(restored.resolve()):raise RuntimeError('Unsafe archive member')
            if member.name!='bundle_manifest.json' and member.name not in inventory:raise RuntimeError('Unexpected member')
        handle.extractall(restored,filter='data')
    for relative,expected in inventory.items():
        path=restored/relative
        if path.stat().st_size!=expected['bytes'] or file_sha256(path)!=expected['sha256']:raise RuntimeError('Restored file mismatch: '+relative)
    env={k:v for k,v in os.environ.items() if not k.startswith('AHR')}
    env.update(AHRV3_WORKSPACE=str(restored),AHRV3_RESTORE_RUNS=json.dumps(list(run_ids)))
    log=v3_report_path(project,'bundle_restore_'+label+'.log')
    with log.open('a') as output:
        result=subprocess.run([sys.executable,'-m','jupyter','nbconvert','--execute','--to','notebook','--output','restore_inference_executed.ipynb','--output-dir',str(restored/'reports/ahr_jepa_v3'),
            '--ExecutePreprocessor.kernel_name=ahr-go','--ExecutePreprocessor.timeout=900',str(restored/V3_ROOT/'restore_inference.ipynb')],
            cwd=restored,env=env,stdout=output,stderr=subprocess.STDOUT)
    if result.returncode:raise RuntimeError('Fresh restored inference failed; inspect '+str(log))
    inference=v3_read_json(restored/'reports/ahr_jepa_v3/restore_inference.json')
    if inference['status']!='PASS' or inference['run_ids']!=list(run_ids):raise RuntimeError('Restore receipt mismatch')
    receipt={'status':'VERIFIED_LOCAL','archive':str(archive.relative_to(project)),'archive_sha256':digest_archive,
             'restored_workspace':str(restored.relative_to(project)),'run_ids':list(run_ids),'files':len(inventory),
             'bytes_uncompressed':sum(e['bytes'] for e in inventory.values()),'restore_inference':inference,
             'off_instance_backup':'DEFERRED_BY_USER','controller_restart_scope':'Use protocol notebook with original checkpoint run; this bundle stores completed scientific runs, not a running controller PID.'}
    atomic_json(receipt_path,receipt)
    return receipt


In [ ]:
print('READY: local experiment backup is executed by the controller after all scientific writers finish.')